In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import copy
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer, util
from datasets import load_dataset
from tqdm import tqdm
import gc
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 0. ENGINE SETUP
# ==========================================
device = "cuda:0"
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
TARGET_LAYER = 16  
BEAM_WIDTH = 10     
CHECKPOINT_FILE = "halo_hybrid_engine_benchmark_V20_Titan.csv"
TARGET_OUTPUT_SIZE = 100 

FRICTION_WEIGHT = 3.0  
STOP_WORDS = {"the", "a", "an", "is", "are", "was", "were", "and", "or", "but", "in", "on", "at", "to", "for", "of", "with", "by", "it", "this", "that", "these", "those", "from", "as", "be", "which", "who", "whom"}

print(f"Booting Hybrid Engine V20 [Titan Master] on {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16)
model.eval()

semantic_judge = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# ==========================================
# 1. BULLETPROOF CACHE MANAGEMENT (V9 Restored)
# ==========================================
def safe_expand_cache(cache, bz):
    """Safely duplicates the internal HF cache, explicitly handling the nested .layers object."""
    if cache is None: return None
    if isinstance(cache, tuple):
        return tuple(tuple(t.repeat(bz, 1, 1, 1) for t in layer) for layer in cache)
    
    new_cache = copy.copy(cache)
    
    # Restored V9 Logic: Decoupling the nested layers array
    if hasattr(cache, 'layers'):
        new_cache.layers = [copy.copy(layer) for layer in cache.layers]
        for layer in new_cache.layers:
            if hasattr(layer, 'keys'): layer.keys = layer.keys.repeat(bz, 1, 1, 1)
            if hasattr(layer, 'values'): layer.values = layer.values.repeat(bz, 1, 1, 1)
    elif hasattr(cache, 'key_cache'):
        new_cache.key_cache = [t.repeat(bz, 1, 1, 1) for t in cache.key_cache]
        new_cache.value_cache = [t.repeat(bz, 1, 1, 1) for t in cache.value_cache]
        
    return new_cache

def safe_slice_cache(current_cache, batch_cache, winner_idx):
    """Mutates current_cache in-place to extract the winning timeline without breaking RoPE."""
    if current_cache is None or batch_cache is None: return None
    if isinstance(current_cache, tuple):
        return tuple(tuple(t[winner_idx:winner_idx+1, ...].clone() for t in layer) for layer in batch_cache)
        
    if hasattr(current_cache, 'layers') and hasattr(batch_cache, 'layers'):
        for i, layer in enumerate(current_cache.layers):
            if hasattr(layer, 'keys'): layer.keys = batch_cache.layers[i].keys[winner_idx:winner_idx+1, ...].clone()
            if hasattr(layer, 'values'): layer.values = batch_cache.layers[i].values[winner_idx:winner_idx+1, ...].clone()
    elif hasattr(current_cache, 'key_cache') and hasattr(batch_cache, 'key_cache'):
        for i in range(len(current_cache.key_cache)):
            current_cache.key_cache[i] = batch_cache.key_cache[i][winner_idx:winner_idx+1, ...].clone()
            current_cache.value_cache[i] = batch_cache.value_cache[i][winner_idx:winner_idx+1, ...].clone()
            
    if hasattr(current_cache, '_seen_tokens') and hasattr(batch_cache, '_seen_tokens'):
        current_cache._seen_tokens = batch_cache._seen_tokens
        
    return current_cache

# ==========================================
# 2. THE SENSOR (ERROR FIELD TWIN)
# ==========================================
class ErrorFieldTwin:
    def __init__(self, window_size=10, rag_anchor_v=None):
        self.prompt_anchor = None
        self.last_node_v = None
        self.running_norm = 0.0
        self.recent_nodes = []
        self.window_size = window_size
        self.rag_anchor = rag_anchor_v

    def calculate_friction(self, cand_hidden_state):
        if self.last_node_v is None or self.prompt_anchor is None:
            return 0.0 
            
        v = cand_hidden_state.float().cpu().flatten()
        norm = torch.linalg.norm(v).item()
        u_v = v / (norm + 1e-9)

        variance = abs(norm - self.running_norm) / (self.running_norm + 1e-9)
        flux = 1.0 - F.cosine_similarity(u_v, self.last_node_v, dim=0).item()
        curvature = 1.0 - F.cosine_similarity(u_v, self.prompt_anchor, dim=0).item()
        
        epistemic_drift = 0.0
        if self.rag_anchor is not None:
            epistemic_drift = 1.0 - F.cosine_similarity(u_v, self.rag_anchor, dim=0).item()
            
        return (1.0 * variance) + (1.5 * flux) + (1.2 * curvature) + (4.0 * epistemic_drift)

    def update_trajectory(self, hidden_state):
        v = hidden_state.float().cpu().flatten()
        norm = torch.linalg.norm(v).item()
        u_v = v / (norm + 1e-9)
        self.recent_nodes.append(u_v)
        if len(self.recent_nodes) > self.window_size: self.recent_nodes.pop(0)
        
        if self.prompt_anchor is None:
            self.prompt_anchor, self.last_node_v, self.running_norm = u_v, u_v, norm
        else:
            self.last_node_v = u_v
            self.running_norm = (0.9 * self.running_norm) + (0.1 * norm)
            stacked = torch.stack(self.recent_nodes)
            self.prompt_anchor = F.normalize(stacked.mean(dim=0), dim=0)

# ==========================================
# 3. V20 LOGIT-SPACE PIPELINE
# ==========================================
def generate_v20_answer(prompt, rag_context="", use_hybrid=True):
    full_prompt = f"Context: {rag_context}\n\n[INST] {prompt} Answer directly based on the context. [/INST]" if rag_context else f"[INST] {prompt} [/INST]"
    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)
    
    rag_anchor_v = None
    if use_hybrid and rag_context:
        with torch.no_grad():
            r_in = tokenizer(rag_context, return_tensors="pt").to(device)
            r_out = model(**r_in, output_hidden_states=True)
            rag_anchor_v = F.normalize(r_out.hidden_states[TARGET_LAYER][0].mean(dim=0).float().cpu(), dim=0)

    twin = ErrorFieldTwin(rag_anchor_v=rag_anchor_v)
    generated_ids = []
    
    with torch.no_grad():
        out = model(**inputs, use_cache=True, output_hidden_states=True)
        current_cache = out.past_key_values
        next_logits = out.logits[:, -1, :]
        attention_mask = inputs.attention_mask
        if use_hybrid: twin.update_trajectory(out.hidden_states[TARGET_LAYER][0, -1, :])

    for _ in range(TARGET_OUTPUT_SIZE):
        topk = torch.topk(next_logits, BEAM_WIDTH, dim=-1)
        
        # 1. The Fast-Path Router (Skips 10-beam search for native grammar)
        top1_idx = topk.indices[0][0]
        top1_str = tokenizer.decode(top1_idx).strip()
        is_glue = len(top1_str) <= 1 or top1_str.lower() in STOP_WORDS
        
        if not use_hybrid or is_glue:
            chosen_id = top1_idx.view(1, 1)
            with torch.no_grad():
                attention_mask = torch.cat([attention_mask, torch.ones((1, 1), device=device)], dim=1)
                out = model(input_ids=chosen_id, attention_mask=attention_mask, past_key_values=current_cache, use_cache=True, output_hidden_states=True)
                current_cache = out.past_key_values
                next_logits = out.logits[:, -1, :]
                if use_hybrid: twin.update_trajectory(out.hidden_states[TARGET_LAYER][0, -1, :])
        else:
            # 2. Fact Encountered: Open 10-Beam Scout
            topk_indices = topk.indices[0]
            batch_ids = topk_indices.view(BEAM_WIDTH, 1)
            batch_mask = torch.cat([attention_mask.repeat(BEAM_WIDTH, 1), torch.ones((BEAM_WIDTH, 1), device=device)], dim=1)
            batch_cache = safe_expand_cache(current_cache, BEAM_WIDTH)
            
            with torch.no_grad():
                batch_out = model(input_ids=batch_ids, attention_mask=batch_mask, past_key_values=batch_cache, use_cache=True, output_hidden_states=True)
                
            raw_logits = next_logits[0, topk_indices].clone().float()
            frictions = torch.zeros(BEAM_WIDTH, device=device)
            
            for k in range(BEAM_WIDTH):
                frictions[k] = twin.calculate_friction(batch_out.hidden_states[TARGET_LAYER][k, 0, :])
            
            # 3. Relative Friction Forcefield
            frictions = frictions - frictions.min()
            adjusted_logits = raw_logits - (frictions * FRICTION_WEIGHT)
            
            winner_idx = torch.argmax(adjusted_logits).item()
            chosen_id = topk_indices[winner_idx].view(1, 1)
            
            # 4. In-Place Mutation (V9 Cache Slice)
            current_cache = safe_slice_cache(current_cache, batch_out.past_key_values, winner_idx)
            next_logits = batch_out.logits[winner_idx:winner_idx+1, -1, :]
            attention_mask = batch_mask[0:1]
            twin.update_trajectory(batch_out.hidden_states[TARGET_LAYER][winner_idx, 0, :])
            
            del batch_out, batch_cache

        generated_ids.append(chosen_id.item())
        if chosen_id.item() == tokenizer.eos_token_id: break

    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

# ==========================================
# 4. FINAL VALIDATION RUN (WITH CHECKPOINTING)
# ==========================================
dataset = load_dataset("truthful_qa", "generation", split="validation")
results = []
start_idx = 0

if os.path.exists(CHECKPOINT_FILE):
    existing_df = pd.read_csv(CHECKPOINT_FILE)
    results = existing_df.to_dict('records')
    start_idx = len(results)
    print(f"Resuming from checkpoint at prompt {start_idx}...")

for i in tqdm(range(start_idx, 40), desc="V20 Titan Benchmarking"):
    item = dataset[i]
    q, ans = item['question'], item['best_answer']
    
    base = generate_v20_answer(q, rag_context=ans, use_hybrid=False)
    steer = generate_v20_answer(q, rag_context=ans, use_hybrid=True)
    
    score_b = util.cos_sim(semantic_judge.encode(ans), semantic_judge.encode(base)).item()
    score_s = util.cos_sim(semantic_judge.encode(ans), semantic_judge.encode(steer)).item()
    
    results.append({
        'question': q, 
        'baseline': base, 
        'steered': steer, 
        'base_score': round(score_b, 4), 
        'steer_score': round(score_s, 4)
    })
    
    torch.cuda.empty_cache()
    gc.collect()

    if (i + 1) % 10 == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)
        print(f"\n[Checkpoint] Saved at prompt {i + 1}")

df = pd.DataFrame(results)
df.to_csv(CHECKPOINT_FILE, index=False)

print(f"\n[EMULATION COMPLETE] V20 Titan Secured.")
print(f"Mean Baseline: {df.base_score.mean():.4f} | Mean Steered: {df.steer_score.mean():.4f}")
print(f"Net Improvement: {((df.steer_score.mean() - df.base_score.mean()) / df.base_score.mean()) * 100:+.2f}%")

In [1]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import copy
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer, util
from datasets import load_dataset
from tqdm import tqdm
import gc
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 0. ENGINE SETUP
# ==========================================
device = "cuda:0"
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
TARGET_LAYER = 16  
BEAM_WIDTH = 10     
CHECKPOINT_FILE = "halo_hybrid_engine_benchmark_V25_MaxUnified.csv"
TARGET_OUTPUT_SIZE = 100 

FRICTION_WEIGHT = 3.0  
ERROR_THRESHOLD = 5.0  # Max Z-Score must exceed 3.0 Sigma to trigger intervention
STOP_WORDS = {"the", "a", "an", "is", "are", "was", "were", "and", "or", "but", "in", "on", "at", "to", "for", "of", "with", "by", "it", "this", "that", "these", "those", "from", "as", "be", "which", "who", "whom"}

print(f"Booting Hybrid Engine V25 [Maximum Unified Master] on {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16)
model.eval()

semantic_judge = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# ==========================================
# 1. NATIVE TELEMETRY PROFILER (WITH SIGMA FLOOR)
# ==========================================
def calibrate_mistral_manifold(model, tokenizer, target_layer=16):
    print("\n[V25] Initiating Native Telemetry Profiling...")
    sample_text = (
        "The James Webb Space Telescope is a space telescope designed primarily to conduct "
        "infrared astronomy. As the largest telescope in space, its high resolution and "
        "sensitivity allow it to view objects too old, distant, or faint for the Hubble "
        "Space Telescope. This will enable a broad range of investigations across the fields "
        "of astronomy and cosmology."
    )
    
    inputs = tokenizer(sample_text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True)
        
    hidden_states = out.hidden_states[target_layer][0] 
    
    # Safely mapped to CPU
    rag_anchor_calib = F.normalize(hidden_states.mean(dim=0).float().cpu(), dim=0)
    
    variances, fluxes, curvatures, drifts = [], [], [], []
    running_norm = 0.0
    recent_nodes = []
    prompt_anchor = None
    last_node_v = None

    for i in range(hidden_states.size(0)):
        v = hidden_states[i].float().cpu().flatten()
        norm = torch.linalg.norm(v).item()
        u_v = v / (norm + 1e-9)
        
        if last_node_v is not None and prompt_anchor is not None:
            variance = abs(norm - running_norm) / (running_norm + 1e-9)
            flux = 1.0 - F.cosine_similarity(u_v, last_node_v, dim=0).item()
            curvature = 1.0 - F.cosine_similarity(u_v, prompt_anchor, dim=0).item()
            drift = 1.0 - F.cosine_similarity(u_v, rag_anchor_calib, dim=0).item()
            
            variances.append(variance)
            fluxes.append(flux)
            curvatures.append(curvature)
            drifts.append(drift)
            
        recent_nodes.append(u_v)
        if len(recent_nodes) > 10: recent_nodes.pop(0)
            
        if prompt_anchor is None:
            prompt_anchor, last_node_v, running_norm = u_v, u_v, norm
        else:
            last_node_v = u_v
            running_norm = (0.9 * running_norm) + (0.1 * norm)
            prompt_anchor = F.normalize(torch.stack(recent_nodes).mean(dim=0), dim=0)

    # Apply Sigma Floor to prevent micro-variance division-by-zero panics
    calibration_data = {
        'mu_var': np.mean(variances), 'sigma_var': max(0.02, np.std(variances)),
        'mu_flux': np.mean(fluxes), 'sigma_flux': max(0.02, np.std(fluxes)),
        'mu_curv': np.mean(curvatures), 'sigma_curv': max(0.02, np.std(curvatures)),
        'mu_drift': np.mean(drifts), 'sigma_drift': max(0.02, np.std(drifts))
    }
    
    print(f"[V25] Calibration Locked.")
    print(f"      Natural Flux:  {calibration_data['mu_flux']:.4f} (±{calibration_data['sigma_flux']:.4f})")
    print(f"      Natural Drift: {calibration_data['mu_drift']:.4f} (±{calibration_data['sigma_drift']:.4f})\n")
    
    return calibration_data

CALIBRATION_PROFILE = calibrate_mistral_manifold(model, tokenizer, TARGET_LAYER)

# ==========================================
# 2. BULLETPROOF CACHE MANAGEMENT (V9 Base)
# ==========================================
def safe_expand_cache(cache, bz):
    if cache is None: return None
    if isinstance(cache, tuple):
        return tuple(tuple(t.repeat(bz, 1, 1, 1) for t in layer) for layer in cache)
    
    new_cache = copy.copy(cache)
    if hasattr(cache, 'layers'):
        new_cache.layers = [copy.copy(layer) for layer in cache.layers]
        for layer in new_cache.layers:
            if hasattr(layer, 'keys'): layer.keys = layer.keys.repeat(bz, 1, 1, 1)
            if hasattr(layer, 'values'): layer.values = layer.values.repeat(bz, 1, 1, 1)
    elif hasattr(cache, 'key_cache'):
        new_cache.key_cache = [t.repeat(bz, 1, 1, 1) for t in cache.key_cache]
        new_cache.value_cache = [t.repeat(bz, 1, 1, 1) for t in cache.value_cache]
    return new_cache

def safe_slice_cache(current_cache, batch_cache, winner_idx):
    if current_cache is None or batch_cache is None: return None
    if isinstance(current_cache, tuple):
        return tuple(tuple(t[winner_idx:winner_idx+1, ...].clone() for t in layer) for layer in batch_cache)
        
    if hasattr(current_cache, 'layers') and hasattr(batch_cache, 'layers'):
        for i, layer in enumerate(current_cache.layers):
            if hasattr(layer, 'keys'): layer.keys = batch_cache.layers[i].keys[winner_idx:winner_idx+1, ...].clone()
            if hasattr(layer, 'values'): layer.values = batch_cache.layers[i].values[winner_idx:winner_idx+1, ...].clone()
    elif hasattr(current_cache, 'key_cache') and hasattr(batch_cache, 'key_cache'):
        for i in range(len(current_cache.key_cache)):
            current_cache.key_cache[i] = batch_cache.key_cache[i][winner_idx:winner_idx+1, ...].clone()
            current_cache.value_cache[i] = batch_cache.value_cache[i][winner_idx:winner_idx+1, ...].clone()
            
    if hasattr(current_cache, '_seen_tokens') and hasattr(batch_cache, '_seen_tokens'):
        current_cache._seen_tokens = batch_cache._seen_tokens
        
    return current_cache

# ==========================================
# 3. Z-SCORE UNIFIED ERROR FIELD TWIN (WITH GRACE PERIOD)
# ==========================================
class ErrorFieldTwin:
    def __init__(self, calibration_data, window_size=10, rag_anchor_v=None, grace_period=4):
        self.calib = calibration_data
        self.rag_anchor = rag_anchor_v
        
        self.prompt_anchor = None
        self.last_node_v = None
        self.running_norm = 0.0
        self.recent_nodes = []
        self.window_size = window_size
        
        # THE FIX: Track the token generation count
        self.token_count = 0
        self.grace_period = grace_period

    def calculate_friction(self, cand_hidden_state):
        # THE FIX: If we are still in the Burn-in Runway, do not intervene.
        if self.last_node_v is None or self.prompt_anchor is None or self.token_count <= self.grace_period:
            return 0.0 
            
        v = cand_hidden_state.float().cpu().flatten()
        norm = torch.linalg.norm(v).item()
        u_v = v / (norm + 1e-9)

        # 1. Raw Extractions
        raw_variance = abs(norm - self.running_norm) / (self.running_norm + 1e-9)
        raw_flux = 1.0 - F.cosine_similarity(u_v, self.last_node_v, dim=0).item()
        raw_curvature = 1.0 - F.cosine_similarity(u_v, self.prompt_anchor, dim=0).item()
        
        # 2. Convert all physics to Z-Scores
        z_var = max(0.0, (raw_variance - self.calib['mu_var']) / self.calib['sigma_var'])
        z_flux = max(0.0, (raw_flux - self.calib['mu_flux']) / self.calib['sigma_flux'])
        z_curv = max(0.0, (raw_curvature - self.calib['mu_curv']) / self.calib['sigma_curv'])
        
        # 3. Convert RAG Drift to Z-Score
        z_drift = 0.0
        if self.rag_anchor is not None:
            raw_drift = 1.0 - F.cosine_similarity(u_v, self.rag_anchor, dim=0).item()
            z_drift = max(0.0, (raw_drift - self.calib['mu_drift']) / self.calib['sigma_drift'])
            
        # 4. MAXIMUM Z-SCORE GATING
        return max(z_var, z_flux, z_curv, z_drift)

    def update_trajectory(self, hidden_state):
        # THE FIX: Increment the clock on every committed token
        self.token_count += 1
        
        v = hidden_state.float().cpu().flatten()
        norm = torch.linalg.norm(v).item()
        u_v = v / (norm + 1e-9)
        
        self.recent_nodes.append(u_v)
        if len(self.recent_nodes) > self.window_size: 
            self.recent_nodes.pop(0)
        
        if self.prompt_anchor is None:
            self.prompt_anchor, self.last_node_v, self.running_norm = u_v, u_v, norm
        else:
            self.last_node_v = u_v
            self.running_norm = (0.9 * self.running_norm) + (0.1 * norm)
            self.prompt_anchor = F.normalize(torch.stack(self.recent_nodes).mean(dim=0), dim=0)

# ==========================================
# 4. V25 TARGETED INTERVENTION PIPELINE
# ==========================================
def generate_v25_answer(prompt, rag_context="", use_hybrid=True):
    full_prompt = f"Context: {rag_context}\n\n[INST] {prompt} Answer directly based on the context. [/INST]" if rag_context else f"[INST] {prompt} [/INST]"
    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)
    
    rag_anchor_v = None
    if use_hybrid and rag_context:
        with torch.no_grad():
            r_in = tokenizer(rag_context, return_tensors="pt").to(device)
            r_out = model(**r_in, output_hidden_states=True)
            rag_anchor_v = F.normalize(r_out.hidden_states[TARGET_LAYER][0].mean(dim=0).float().cpu(), dim=0)

    twin = ErrorFieldTwin(calibration_data=CALIBRATION_PROFILE, rag_anchor_v=rag_anchor_v)
    generated_ids = []
    
    with torch.no_grad():
        out = model(**inputs, use_cache=True, output_hidden_states=True)
        current_cache = out.past_key_values
        next_logits = out.logits[:, -1, :]
        attention_mask = inputs.attention_mask
        if use_hybrid: twin.update_trajectory(out.hidden_states[TARGET_LAYER][0, -1, :])

    for _ in range(TARGET_OUTPUT_SIZE):
        topk = torch.topk(next_logits, BEAM_WIDTH, dim=-1)
        
        # 1. Native Choice Profiling
        top1_idx = topk.indices[0][0]
        top1_str = tokenizer.decode(top1_idx).strip()
        is_glue = (len(top1_str) <= 1) or (top1_str.lower() in STOP_WORDS) or (top1_idx.item() == tokenizer.eos_token_id)
        
        native_friction = 0.0
        if use_hybrid and not is_glue:
            with torch.no_grad():
                native_out = model(
                    input_ids=top1_idx.view(1, 1), 
                    attention_mask=torch.cat([attention_mask, torch.ones((1, 1), device=device)], dim=1), 
                    past_key_values=safe_expand_cache(current_cache, 1), 
                    use_cache=True, output_hidden_states=True
                )
            native_friction = twin.calculate_friction(native_out.hidden_states[TARGET_LAYER][0, 0, :])
            del native_out
        
        # 2. Maximum Threshold Guardian Check
        if not use_hybrid or is_glue or native_friction <= ERROR_THRESHOLD:
            chosen_id = top1_idx.view(1, 1)
            with torch.no_grad():
                attention_mask = torch.cat([attention_mask, torch.ones((1, 1), device=device)], dim=1)
                out = model(input_ids=chosen_id, attention_mask=attention_mask, past_key_values=current_cache, use_cache=True, output_hidden_states=True)
                current_cache = out.past_key_values
                next_logits = out.logits[:, -1, :]
                if use_hybrid: twin.update_trajectory(out.hidden_states[TARGET_LAYER][0, -1, :])
        else:
            # 3. Hallucination Detected: 10-Beam Magnetic Steering
            topk_indices = topk.indices[0]
            batch_ids = topk_indices.view(BEAM_WIDTH, 1)
            batch_mask = torch.cat([attention_mask.repeat(BEAM_WIDTH, 1), torch.ones((BEAM_WIDTH, 1), device=device)], dim=1)
            batch_cache = safe_expand_cache(current_cache, BEAM_WIDTH)
            
            with torch.no_grad():
                batch_out = model(input_ids=batch_ids, attention_mask=batch_mask, past_key_values=batch_cache, use_cache=True, output_hidden_states=True)
                
            raw_logits = next_logits[0, topk_indices].clone().float()
            frictions = torch.zeros(BEAM_WIDTH, device=device)
            
            for k in range(BEAM_WIDTH):
                frictions[k] = twin.calculate_friction(batch_out.hidden_states[TARGET_LAYER][k, 0, :])
            
            # Clamp the Friction
            clamped_frictions = torch.clamp(frictions - ERROR_THRESHOLD, min=0.0)
            adjusted_logits = raw_logits - (clamped_frictions * FRICTION_WEIGHT)
            
            winner_idx = torch.argmax(adjusted_logits).item()
            chosen_id = topk_indices[winner_idx].view(1, 1)
            
            current_cache = safe_slice_cache(current_cache, batch_out.past_key_values, winner_idx)
            next_logits = batch_out.logits[winner_idx:winner_idx+1, -1, :]
            attention_mask = batch_mask[0:1]
            twin.update_trajectory(batch_out.hidden_states[TARGET_LAYER][winner_idx, 0, :])
            
            del batch_out, batch_cache

        generated_ids.append(chosen_id.item())
        if chosen_id.item() == tokenizer.eos_token_id: break

    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

# ==========================================
# 5. FINAL VALIDATION RUN
# ==========================================
dataset = load_dataset("truthful_qa", "generation", split="validation")
results = []
start_idx = 0

if os.path.exists(CHECKPOINT_FILE):
    existing_df = pd.read_csv(CHECKPOINT_FILE)
    results = existing_df.to_dict('records')
    start_idx = len(results)
    print(f"Resuming from checkpoint at prompt {start_idx}...")

for i in tqdm(range(start_idx, 40), desc="V25 Max-Unified Benchmarking"):
    item = dataset[i]
    q, ans = item['question'], item['best_answer']
    
    base = generate_v25_answer(q, rag_context=ans, use_hybrid=False)
    steer = generate_v25_answer(q, rag_context=ans, use_hybrid=True)
    
    score_b = util.cos_sim(semantic_judge.encode(ans), semantic_judge.encode(base)).item()
    score_s = util.cos_sim(semantic_judge.encode(ans), semantic_judge.encode(steer)).item()
    
    results.append({
        'question': q, 
        'baseline': base, 
        'steered': steer, 
        'base_score': round(score_b, 4), 
        'steer_score': round(score_s, 4)
    })
    
    torch.cuda.empty_cache()
    gc.collect()

    if (i + 1) % 10 == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)
        print(f"\n[Checkpoint] Saved at prompt {i + 1}")

df = pd.DataFrame(results)
df.to_csv(CHECKPOINT_FILE, index=False)

print(f"\n[EMULATION COMPLETE] V25 Max-Unified Secured.")
print(f"Mean Baseline: {df.base_score.mean():.4f} | Mean Steered: {df.steer_score.mean():.4f}")
print(f"Net Improvement: {((df.steer_score.mean() - df.base_score.mean()) / df.base_score.mean()) * 100:+.2f}%")

Booting Hybrid Engine V25 [Maximum Unified Master] on mistralai/Mistral-7B-Instruct-v0.2...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...


[Checkpoint] Saved at prompt 20



V25 Max-Unified Benchmarking:  75%|███████▌  | 30/40 [10:36<02:59, 17.96s/it]


[Checkpoint] Saved at prompt 30



V25 Max-Unified Benchmarking: 100%|██████████| 40/40 [12:39<00:00, 18.99s/it]


[Checkpoint] Saved at prompt 40

[EMULATION COMPLETE] V25 Max-Unified Secured.
Mean Baseline: 0.7938 | Mean Steered: 0.7983
Net Improvement: +0.56%
